Imports

In [53]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import time

Train and test split

In [54]:
def train_test_split (df, i=0) :
    train = df.sample(frac=0.8, random_state=i)
    test = df.drop(train.index)
    xtrain = np.array(train.drop(columns=[0]))
    xtest = np.array(test.drop(columns=[0]))
    ytrain = np.array(train[0])
    ytest = np.array(test[0])
    return xtrain, xtest, ytrain, ytest

Kernel Functions

In [55]:
def guass_ker_matrix(x1, x2):
    x1sq = np.sum(x1 ** 2, axis=1)[:, None]  # (n1, 1)
    x2sq = np.sum(x2 ** 2, axis=1)[None, :]  # (1, n2)
    num = x1sq + x2sq - 2 * (x1 @ x2.T)
    return (num)
def guass_kern (x1, x2, c=1, **kwargs):
    #diff = np.array(x1) - np.array(x2)
    x1sq = np.sum(x1 ** 2, axis=1)[:, None]  # (n1, 1)
    x2sq = np.sum(x2 ** 2, axis=1)[None, :]  # (1, n2)
    num = x1sq + x2sq -2*(x1@x2.T)
    # num = guass_ker_matrix(x1, x2)
    # diff = x1 - x2
    # num = norm_p(diff, 2, axis=-1)**2
    K = np.exp(-c * (num))
    return K
def poly_kern_matrix (x1, x2):
    return (x1@x2.T)
def poly_kern (x1, x2, d=1, **kwargs) :
    # matrix = poly_kern_matrix(x1, x2)
    # return (matrix)**d
    return (x1@x2.T) ** d

def block_kern (x1, x2, n1, n2, kernel_func, bSize = 250, **kwargs):
    noBlocks1 = int(np.ceil(n1 / bSize))
    noBlocks2 = int(np.ceil(n2 / bSize))
    blocks1 = []
    blocks2 = []
    for b in range(noBlocks1):
        start = b * bSize
        end = min((b + 1) * bSize, n1)
        blocks1.append((start, end))

    for b in range(noBlocks2):
        start = b * bSize
        end = min((b + 1) * bSize, n2)
        blocks2.append((start, end))
    K = np.zeros((n1, n2))
    for iStart, iEnd in blocks1:
        for jStart, jEnd in blocks2:
            K[iStart:iEnd, jStart:jEnd] = kernel_func(x1[iStart:iEnd, :], x2[jStart:jEnd, :], **kwargs)
    return K


def full_kern (x1, kernel_func, x2=None, bSize=250, **kwargs) :
    x1 = np.array(x1)
    n1 =len(x1)
    if x2 is None :
        x2 = x1
        n2 = n1
    else:
        x2 = np.array(x2)
        n2 = len(x2)

    # x1 = x1[:, None, :] #Make 3D and stitch together features dimension
    # x2 = x2[None, :, :]
    if n1 > bSize:
        K = block_kern(x1, x2, n1, n2, kernel_func=kernel_func, **kwargs)
    else :
        K = kernel_func(x1, x2, **kwargs) #Return full Kernel
    return K

Two class classifier functions

In [56]:
def two_class_ker_perc_pred (alpha, x, y, target_class, K_precal_test):
    y_pred = []
    total_errors = 0
    for i in range(len(x)):
        K = K_precal_test[i, :]
        # K = K_precal[:,i]
        K = np.array(K)
        if (alpha @ K) >= 0:
            y_hat = 1
        else:
            y_hat = -1
        if y_hat != y[i]:
            total_errors = total_errors + 1
        y_pred.append(y_hat)
    total_errors = total_errors / len(x)
    return y_pred, total_errors

def two_class_ker_perc_full (df, kernel_func, target_class, epoch = 1, K_precal_train = None, K_precal_test=None, split = None, **kwargs):
    if split is None : #If no prespecified train test splits given make from scratch
        xtrain, xtest, ytrain, ytest = train_test_split(df)
    else : #Otherwise if prespecified train test splits given unpack
        xtrain, xtest, ytrain, ytest = split
    ytrain = np.where(ytrain == target_class, 1, -1)  # Change ytrain 1 = target_class, -1 otherwise
    ytest = np.where(ytest == target_class, 1, -1)  # Same for ytest
    epoch_threshold = 5
    no_improvement_count = 0
    min_epoch_errors = np.inf
    alpha = np.zeros(len(xtrain))
    best_alpha = alpha.copy()
    if K_precal_train is None :
        # print('Pre computing Train Kernel')
        K_precal_train = full_kern(xtrain, kernel_func=kernel_func, **kwargs)
        # print('Train Kernel Computed')
    total_errors_train = 0
    for e in range(epoch) :
        epoch_errors = 0
        #print(f"Epoch {e}, Class = {target_class}")
        for i in range(len(xtrain)) :
            alpha_i = alpha[:i]  # all previous alpha's
            K = K_precal_train[:i, i]
            if (alpha_i @ K) >= 0:
                y_hat = 1
            else :
                y_hat = -1
            if y_hat != ytrain[i] :
                alpha[i] = alpha[i] + ytrain[i]
                epoch_errors = epoch_errors + 1
                total_errors_train = total_errors_train + 1
        if epoch_errors < min_epoch_errors:
            min_epoch_errors = epoch_errors
            no_improvement_count = 0
            best_alpha = alpha.copy()
        else :
            no_improvement_count = no_improvement_count + 1
        if no_improvement_count >= epoch_threshold:
            #print(f"Stopped early at epoch {e} for class {target_class}")
            break
    total_errors_train = min_epoch_errors / len(xtrain)

    alpha = best_alpha.copy()
    if K_precal_test is None :
        # print('Pre Computing Test Kernel')
        K_precal_test = full_kern(xtest, kernel_func=kernel_func, x2=xtrain, **kwargs)
        # print('Test Kernel Computed')
        y_pred_test, total_errors_test = two_class_ker_perc_pred(alpha, xtest, ytest, target_class, K_precal_test)
        return alpha, total_errors_train, y_pred_test, total_errors_test
    # y_pred = []
    # total_errors = 0
    # alpha = np.array(alpha)
    # for i in range(len(xtest)) :
    #     K = K_precal_test[i, :]
    #     # K = K_precal[:,i]
    #     K = np.array(K)
    #     if (alpha @ K) >= 0:
    #         y_hat = 1
    #     else:
    #         y_hat = -1
    #     if y_hat != ytest[i]:
    #         total_errors = total_errors + 1
    #     y_pred.append(y_hat)
    # total_errors = total_errors/len(xtest)
    return alpha, total_errors_train, None, None

k class classifier functions

In [57]:
def k_class_ker_perc_pred (alpha, x, y, k_classes, K_precal_test):
    y_pred = []
    total_errors = 0
    for i in range(len(x)) :
        y_hat = []
        K = K_precal_test[i, :]
        # K = K_precal[:,i]
        K = np.array(K)
        for j, k in enumerate (k_classes) :
            y_hat.append(alpha[j] @ K)
        y_pred.append(k_classes[y_hat.index(max(y_hat))])
        if y_pred[i] != y[i]:
            total_errors = total_errors + 1
    total_errors = total_errors/len(x)
    return y_pred, total_errors

def k_class_ker_perc (df, kernel_func, k_classes, epoch=1, split=None, K_precal_train = None, K_precal_test = None, **kwargs):
    if split is None : xtrain, xtest, ytrain, ytest = train_test_split(df) #create test train splits
    else : xtrain, xtest, ytrain, ytest = split
    alpha, y_pred_test, train_errors, test_erros = [], [], [], []
    if K_precal_train is None:
        # print('pre computing  train kernel')
        K_precal_train = full_kern(xtrain, kernel_func=kernel_func, **kwargs) #Precalculate the Kernel Matrix
        # print('train kernel computed')
    if K_precal_test is None:
        # print('pre computing test kernel')
        K_precal_test = full_kern(xtest, kernel_func=kernel_func, x2=xtrain, **kwargs)
        # print('test kernel computed')
        # print('kernels computed')
    for k in k_classes : #Loop through the k classes
        split = (xtrain, xtest, ytrain.copy(), ytest.copy()) #Store new dataset with y {1,-1} labels
        alpha_k, train_errors_k, _, _ = two_class_ker_perc_full(df, kernel_func, k, epoch=epoch, K_precal_train=K_precal_train, K_precal_test=K_precal_test, split = split) #Train 2 class classifier, (k vs rest)
        alpha.append(alpha_k) #Add k vs rest alpha values to alpha list
        train_errors.append(train_errors_k)
    y_pred_test, test_errors = k_class_ker_perc_pred(alpha, xtest, ytest, k_classes, K_precal_test)
    me_train = np.mean(train_errors)
    me_test = np.mean(test_errors)
    return alpha, y_pred_test, me_train, me_test, train_errors, test_errors

Cross Validation

In [58]:
def cross_validation(xtrain, ytrain, folds, kernel_func, paramlist, k_classes, epoch_lim=1, **kwargs):
    positions = np.arange(len(xtrain))
    np.random.shuffle(positions)
    foldPositions = np.array_split(positions, folds)


    K_precal = {p: full_kern(xtrain, kernel_func=kernel_func, d=p, c=p) for p in paramlist}


    paraMatrix = paramlist

    cross_results = []
    results = np.zeros((folds, len(paramlist), 2))
    for j, p in enumerate(paramlist):
        K_full = K_precal[p]

        for i in range(folds):

            xtrain_fold = np.concatenate([foldPositions[f] for f in range(folds) if f != i])
            xtest_fold = foldPositions[i]


            K_train = K_full[np.ix_(xtrain_fold, xtrain_fold)]
            K_test = K_full[np.ix_(xtest_fold, xtrain_fold)]
            ytrain_fold = ytrain[xtrain_fold]
            ytest_fold = ytrain[xtest_fold]
            xtrain_fold = xtrain[xtrain_fold]
            xtest_fold = xtrain[xtest_fold]
            split_fold = (xtrain_fold, xtest_fold, ytrain_fold, ytest_fold)

            alpha, y_pred_test, me_train, me_test, _, _ = k_class_ker_perc(None, kernel_func, k_classes, epoch=epoch_lim, split=split_fold, K_precal_train = K_train, K_precal_test = K_test, d=p, c=p)
            results[i, j, 0] = me_train
            results[i, j, 1] = me_test

            # alpha = ker_ridg_sol(K_train, gam, ytrain_fold)
            # y_hat_train = K_train @ alpha
            # y_hat_test = K_test @ alpha
            # mse_train_fold = np.mean((ytrain_fold - y_hat_train) ** 2)
            # mse_test_fold = np.mean((ytest_fold - y_hat_test) ** 2)
            # mse = [mse_train_fold, mse_test_fold]
            # results.append(mse)
    avgme_train = np.mean(results[:, :, 0], axis=0)
    avgme_test = np.mean(results[:, :, 1], axis=0)
        # avgmse = [0, 0]
        #
        # for i in range(2):
        #     total = 0
        #     for j in range(folds):
        #         total += results[j][i]
        #     avgmse[i] = total / folds

    avgme_df = pd.DataFrame({'Param': paramlist, 'avg_me_train_fold': avgme_train, 'avg_me_test_fold': avgme_test})
    # cross_results = pd.concat(cross_results)
    avgme_df = avgme_df.sort_values(by ='avg_me_test_fold', ascending=True)
    min_me = avgme_df.iloc[0]
    return float(min_me['Param']), min_me['avg_me_train_fold'], min_me['avg_me_test_fold'], avgme_df

r sims

In [59]:
def cross_validation(xtrain, ytrain, folds, kernel_func, paramlist, k_classes, epoch_lim=1, **kwargs):
    positions = np.arange(len(xtrain))
    np.random.shuffle(positions)
    foldPositions = np.array_split(positions, folds)


    K_precal = {p: full_kern(xtrain, kernel_func=kernel_func, d=p, c=p) for p in paramlist}


    paraMatrix = paramlist

    cross_results = []
    results = np.zeros((folds, len(paramlist), 2))
    for j, p in enumerate(paramlist):
        K_full = K_precal[p]

        for i in range(folds):

            xtrain_fold = np.concatenate([foldPositions[f] for f in range(folds) if f != i])
            xtest_fold = foldPositions[i]


            K_train = K_full[np.ix_(xtrain_fold, xtrain_fold)]
            K_test = K_full[np.ix_(xtest_fold, xtrain_fold)]
            ytrain_fold = ytrain[xtrain_fold]
            ytest_fold = ytrain[xtest_fold]
            xtrain_fold = xtrain[xtrain_fold]
            xtest_fold = xtrain[xtest_fold]
            split_fold = (xtrain_fold, xtest_fold, ytrain_fold, ytest_fold)

            alpha, y_pred_test, me_train, me_test, _, _ = k_class_ker_perc(None, kernel_func, k_classes, epoch=epoch_lim, split=split_fold, K_precal_train = K_train, K_precal_test = K_test, d=p, c=p)
            results[i, j, 0] = me_train
            results[i, j, 1] = me_test

            # alpha = ker_ridg_sol(K_train, gam, ytrain_fold)
            # y_hat_train = K_train @ alpha
            # y_hat_test = K_test @ alpha
            # mse_train_fold = np.mean((ytrain_fold - y_hat_train) ** 2)
            # mse_test_fold = np.mean((ytest_fold - y_hat_test) ** 2)
            # mse = [mse_train_fold, mse_test_fold]
            # results.append(mse)
    avgme_train = np.mean(results[:, :, 0], axis=0)
    avgme_test = np.mean(results[:, :, 1], axis=0)
        # avgmse = [0, 0]
        #
        # for i in range(2):
        #     total = 0
        #     for j in range(folds):
        #         total += results[j][i]
        #     avgmse[i] = total / folds

    avgme_df = pd.DataFrame({'Param': paramlist, 'avg_me_train_fold': avgme_train, 'avg_me_test_fold': avgme_test})
    # cross_results = pd.concat(cross_results)
    avgme_df = avgme_df.sort_values(by ='avg_me_test_fold', ascending=True)
    min_me = avgme_df.iloc[0]
    return float(min_me['Param']), min_me['avg_me_train_fold'], min_me['avg_me_test_fold'], avgme_df


# Runs r reps of model fitting, each with different train/test splits, returns the average MSE across runs (train and test)
def r_sims_k_class_ker_perc(df, r, kernel_func, k_classes, param, epoch_lim = 30, splits=None,  crossVal=False, **kwargs) :
    if crossVal == True:
        results = np.zeros((r, 1, 2))
    else:
        results=np.zeros((r, len(param),2))
    fig, ax = None, None

    best_params = []

    for i in range(r):
        if splits is None:
            xtrain, xtest, ytrain, ytest = train_test_split(df, i)
            split = (xtrain, xtest, ytrain.copy(), ytest.copy())
        else:
            split = splits[i]
            xtrain, xtest, ytrain, ytest = split
        if crossVal == True:
            minparam, _, _, crossValdf= cross_validation(xtrain, ytrain, 5, kernel_func, param, k_classes, epoch_lim=epoch_lim)

            best_params.append({
                "run": int(i),
                "best_param": float(minparam)
            })
            alpha, y_pred_test, me_train, me_test, _, _ = k_class_ker_perc(None, kernel_func, k_classes, epoch=epoch_lim, split=split, d=minparam, c=minparam)
            results[i, 0, 0] = me_train
            results[i, 0, 1] = me_test
        else :
            minparam = None
            for j, p in enumerate(param):
                # print(f'run {i}, parameter {p}')
        #print(f'For r = {i+1}, minsig = 2**{np.log2(minsig)}, mingam = 2**{np.log2(mingam)}')
                alpha, y_pred_test, me_train, me_test, _, _ = k_class_ker_perc(None, kernel_func, k_classes, epoch = epoch_lim, split=split, d=p, c=p)
        #krm = ker_reg_sol(xtrain, ytrain, kernel_func, sig=minsig, gam=mingam)
                results[i,j,0] = me_train
                results[i,j,1] = me_test

    avgme_train= np.mean(results[:, :, 0], axis=0)
    avgsde_train = np.std(results[:, :, 0], axis=0)
    avgme_test = np.mean(results[:, :, 1], axis=0)
    avgsde_test = np.std(results[:, :, 1], axis=0)

    if crossVal == True:
        avgmse_df = pd.DataFrame({'param': minparam, 'avg_me_train': avgme_train, 'avg_me_test': avgme_test, 'avg_me_train_sd': avgsde_train, 'avg_me_test_sd': avgsde_test})
    else:
        avgmse_df = pd.DataFrame({'param': param, 'avg_me_train': avgme_train, 'avg_me_test': avgme_test, 'avg_me_train_sd': avgsde_train,'avg_me_test_sd': avgsde_test})
    # Save aggregated best params (Q5a) after all runs
    # if r > 1:  # only when doing repeated runs
    #     save_json({"runs": best_params}, "Q5/best_params.json")
    #
    # if plot == True and r>0:
    #     fig, ax = plot_crossVal_mse(crossValdf)
    #     return avgmse_df, alpha, fig, ax, minsig, mingam, crossValdf
    # else:
    #     return avgmse_df, alpha, minsig, mingam
    return avgmse_df


Main

TODO: Epoch-convergence experiment for report

Run a controlled experiment to justify the early-stopping threshold.

Pick three digits (e.g. 0, 5, 8) and evaluate binary perceptron: digit vs rest.

For each digit, run the two_class_ker_perc_curve function for 15 epochs with:

Polynomial kernel (choose a fixed degree, e.g. d = 5)

Collect the training error per epoch and plot the learning curves.

Show that error drops almost entirely within the first 2–3 epochs, then plateaus.

Use this figure + table to justify choosing an early-stopping threshold of 2–3 non-improving epochs.

In [60]:
url = 'http://www0.cs.ucl.ac.uk/staff/M.Herbster/SL/misc/zipcombo.dat'
data = np.loadtxt("zipcombo.dat")
df = pd.DataFrame(data)
dftest = df.sample(frac=0.15, random_state=0).reset_index(drop=True)
#df = pd.read_csv(url)
print(df.head())

print('Q1')

print('Q2')

print('Q3')
runs = 20
d=[1,2,3,4,5,6,7]
Classes = [0,1,2,3,4,5,6,7,8,9]
start = time.time()
avgme_df = r_sims_k_class_ker_perc(dftest, runs, poly_kern, Classes, param=d)
end = time.time()
print(f"\nTotal runtime = {end - start:.2f} seconds ({(end-start)/60:.2f} minutes)")
print(f'Polynomial Kernel : {runs} run')
print(avgme_df)


   0    1    2    3      4      5      6      7      8      9    ...    247  \
0  6.0 -1.0 -1.0 -1.0 -1.000 -1.000 -1.000 -1.000 -0.631  0.862  ...  0.304   
1  5.0 -1.0 -1.0 -1.0 -0.813 -0.671 -0.809 -0.887 -0.671 -0.853  ... -0.671   
2  4.0 -1.0 -1.0 -1.0 -1.000 -1.000 -1.000 -1.000 -1.000 -1.000  ... -1.000   
3  7.0 -1.0 -1.0 -1.0 -1.000 -1.000 -0.273  0.684  0.960  0.450  ... -0.318   
4  3.0 -1.0 -1.0 -1.0 -1.000 -1.000 -0.928 -0.204  0.751  0.466  ...  0.466   

     248    249    250    251    252    253    254    255  256  
0  0.823  1.000  0.482 -0.474 -0.991 -1.000 -1.000 -1.000 -1.0  
1 -0.671 -0.033  0.761  0.762  0.126 -0.095 -0.671 -0.828 -1.0  
2 -1.000 -1.000 -0.109  1.000 -0.179 -1.000 -1.000 -1.000 -1.0  
3  1.000  0.536 -0.987 -1.000 -1.000 -1.000 -1.000 -1.000 -1.0  
4  0.639  1.000  1.000  0.791  0.439 -0.199 -0.883 -1.000 -1.0  

[5 rows x 257 columns]
Q1
Q2
Q3

Total runtime = 14.40 seconds (0.24 minutes)
Polynomial Kernel : 20 run
   param  avg_me_train  avg_m

Q4

In [61]:
print('Q4')
runs = 20
d=[1,2,3,4,5,6,7]
Classes = [0,1,2,3,4,5,6,7,8,9]
start = time.time()
avgme_df = r_sims_k_class_ker_perc(dftest, runs, poly_kern, Classes, param=d, crossVal=True)
end = time.time()
print(f"\nTotal runtime = {end - start:.2f} seconds ({(end-start)/60:.2f} minutes)")
print(f'Polynomial Kernel : {runs} run with cross validation')
print(avgme_df)

Q4

Total runtime = 45.72 seconds (0.76 minutes)
Polynomial Kernel : 20 run with cross validation
   param  avg_me_train  avg_me_test  avg_me_train_sd  avg_me_test_sd
0    6.0      0.043947     0.098566         0.001796        0.017171


Q7

Gaussian kernel parameter choice: brief justification

Stage 1: Coarse scan
Test very wide values
c = {0.01, 0.1, 1, 10, 100, 1000, 10000}.
Only 0.01–1 behave sensibly; larger values collapse.
This identifies the correct order of magnitude.

Stage 2: Medium scan inside viable band
c = {0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1}.
Errors form a U-shape with the best region around 0.03–0.08.
This narrows the search to one small interval.

Stage 3: Log-scale refinement (base 2)
c = 2^{-k}, k = {3,4,5,6,7} → {0.125, 0.0625, 0.03125, 0.015625, 0.0078}.
Best results consistently around 0.03–0.06.

Final schedule for all experiments
Use a refined half-step grid:
c = 2^{-b/2}, b = 6…12 → {0.125, 0.088, 0.0625, 0.044, 0.031, 0.022, 0.0156}.
This covers the optimal region with good resolution and low cost.

Summary
A 3-stage search (coarse → medium → log-scale) showed the optimal Gaussian parameter lies around 0.03–0.06.
Therefore the final grid c = 2^{-b/2}, b = 6…12 was adopted for all later experiments.

b

In [62]:
print('Q7')
print('(b)')
runs = 20
c = [2**(-b/2) for b in range(6, 13)]
Classes = [0,1,2,3,4,5,6,7,8,9]
start = time.time()
avgme_df = r_sims_k_class_ker_perc(dftest, runs, guass_kern, Classes, param=c)
end = time.time()
print(f"\nTotal runtime = {end - start:.2f} seconds ({(end-start)/60:.2f} minutes)")
print(f'Guassian Kernel : {runs} run')
print(avgme_df)

Q7
(b)

Total runtime = 12.29 seconds (0.20 minutes)
Guassian Kernel : 20 run
      param  avg_me_train  avg_me_test  avg_me_train_sd  avg_me_test_sd
0  0.125000      0.057975     0.145878         0.002105        0.017817
1  0.088388      0.055018     0.134409         0.002179        0.016716
2  0.062500      0.050672     0.123297         0.001975        0.016125
3  0.044194      0.046470     0.115412         0.001892        0.015973
4  0.031250      0.043221     0.098925         0.001676        0.019710
5  0.022097      0.040632     0.097849         0.001716        0.018280
6  0.015625      0.040009     0.092294         0.001551        0.018996


c

In [63]:
print('(c)')
runs = 20
c = [2**(-b/2) for b in range(6, 13)]
Classes = [0,1,2,3,4,5,6,7,8,9]
start = time.time()
avgme_df = r_sims_k_class_ker_perc(dftest, runs, guass_kern, Classes, param=c, crossVal=True)
end = time.time()
print(f"\nTotal runtime = {end - start:.2f} seconds ({(end-start)/60:.2f} minutes)")
print(f'Guassian Kernel : {runs} run with cross validation')
print(avgme_df)

(c)

Total runtime = 44.13 seconds (0.74 minutes)
Guassian Kernel : 20 run with cross validation
      param  avg_me_train  avg_me_test  avg_me_train_sd  avg_me_test_sd
0  0.015625      0.040502     0.090681         0.001689        0.016854


In [64]:
alpha, train_errors, y_pred, test_errors =two_class_ker_perc_full(df, guass_kern, 5, 100)
print(f'2 class classifier for target class 1 : {test_errors}')
alpha, y_pred_test, me_train, me_test, train_errors, test_errors = k_class_ker_perc(df, guass_kern, Classes, 100)
print(f'k class classifier for target classes 0-9 : {test_errors}')

2 class classifier for target class 1 : 0.025268817204301075
k class classifier for target classes 0-9 : 0.08010752688172043
